In [ ]:
#@title 0. Instalar dependencias y montar Drive
# Instalar Panoptic API directamente desde GitHub
!pip -q install "git+https://github.com/cocodataset/panopticapi.git"

from google.colab import drive
drive.mount('/content/drive')

# Instalar detectron2 desde fuente
import sys, os, distutils.core

# Detectron2 (fast local install)
!git clone -q 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install -q {' '.join([f"'{x}'" for x in dist.install_requires])}
import sys, os
sys.path.insert(0, os.path.abspath('./detectron2'))

# Otros paquetes
!pip install -q rasterio pyproj fiona matplotlib albumentations tqdm pycocotools pandas scikit-image Pillow



  Preparing metadata (setup.py) ... done
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.7 MB/s eta 0:00:00


In [ ]:
#@title 1. Rutas, constantes y semilla
import os, re, json, glob, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import rasterio
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from pycocotools.coco import COCO

# Semilla
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Rutas base
BASE = Path("/content/drive/MyDrive/juniper_mapper/JuniperMapper")

# PI: splits Train / Val / Test
PI_TRAIN_JSON = str(BASE/"Photo_Interpretation_Data/Train/Annotations/Train.json")
PI_TRAIN_IMG  = str(BASE/"Photo_Interpretation_Data/Train/Images")
PI_VAL_JSON   = str(BASE/"Photo_Interpretation_Data/Val/Annotations/Val.json")
PI_VAL_IMG    = str(BASE/"Photo_Interpretation_Data/Val/Images")
PI_COCO_JSON  = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Test.json")
PI_IMG_DIR    = str(BASE/"Photo_Interpretation_Data/Test/Images")

# Máscaras SEM de Train/Val (binarias 0/1)
PI_TRAIN_MASKS = str(BASE/"Photo_Interpretation_Data/Train/Annotations/Masks")
PI_VAL_MASKS   = str(BASE/"Photo_Interpretation_Data/Val/Annotations/Masks")

#  FW: test externo
FW_COCO_JSON = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/FW.json")
FW_IMG_DIR   = str(BASE/"Field_Work_Data/External_Val_Data/Images")

# Máscaras GT semánticas (binario 0/1) para métricas pixelares / cobertura-densidad
GT_PI = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Masks")
GT_FW = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/Masks")

# Salidas Panoptic FPN
OUT_PAN  = str(BASE/"PanopticFPN/output")
PRED_PI  = f"{OUT_PAN}/Predictions_PI_thetaStar"
PRED_FW  = f"{OUT_PAN}/Predictions_FW_thetaStar"
for d in [OUT_PAN, f"{OUT_PAN}/csv", f"{OUT_PAN}/tables", f"{OUT_PAN}/curves", f"{OUT_PAN}/plots", PRED_PI, PRED_FW]:
    os.makedirs(d, exist_ok=True)

# Bins de tamaño (m²)
SIZE_BINS = [
    ("XS",  0.13,  1.72),
    ("S",   1.72,  3.62),
    ("M",   3.62,  9.08),
    ("L",   9.08, 20.82),
    ("XL", 20.82, 41.06),
    ("XXL",41.06, float("inf")),
]
def size_label(area_m2: float):
    for name, lo, hi in SIZE_BINS:
        if lo <= area_m2 < hi:
            return name
    return "XS"

# GSD fallback (m/px)
GSD_FALLBACK = 0.13

In [ ]:
#@title 2. Registro de datasets (COCO + inyección de semántica) y helpers
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

# COCO nativo (instancias)
register_coco_instances("pi_train_coco", {}, PI_TRAIN_JSON, PI_TRAIN_IMG)
register_coco_instances("pi_val_coco",   {}, PI_VAL_JSON,   PI_VAL_IMG)
register_coco_instances("pi_test_coco",  {}, PI_COCO_JSON,  PI_IMG_DIR)
register_coco_instances("fw_test_coco",  {}, FW_COCO_JSON,  FW_IMG_DIR)

print("Datasets COCO registrados: pi_train_coco, pi_val_coco, pi_test_coco, fw_test_coco")

#  Helpers de GSD / máscaras
def image_gsd_m(geotiff_path, fallback=GSD_FALLBACK):
    try:
        with rasterio.open(geotiff_path) as src:
            tr = src.transform
            if src.crs and src.crs.is_projected:
                px_m = abs(tr.a); py_m = abs(tr.e)
            else:
                mx = 111320.0; my = 110540.0
                px_m = mx * abs(tr.a); py_m = my * abs(tr.e)
            if px_m > 0 and py_m > 0:
                return float((px_m + py_m) / 2.0)
    except Exception:
        pass
    return float(fallback)

def ann_masks_for_img(coco: COCO, img_id: int, H: int, W: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    masks = []
    for a in anns:
        if a.get("iscrowd",0)==1:
            continue
        m = coco.annToMask(a).astype(bool)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        masks.append(m)
    return masks

def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

#  Wrapper para inyectar semántica en Train/Val
def _mask_path_for_image(img_path: str, masks_dir: str):
    name = Path(img_path).name
    base = "Mask_" + (name[len("Img_"):] if name.startswith("Img_") else name)
    mask_name = str(Path(base).with_suffix(".tif"))
    return str(Path(masks_dir) / mask_name)

def _with_semseg(base_name:str, masks_dir:str):
    def _loader():
        ds = DatasetCatalog.get(base_name)
        out = []
        for d in ds:
            d2 = d.copy()
            d2["sem_seg_file_name"] = _mask_path_for_image(d2["file_name"], masks_dir)
            out.append(d2)
        return out
    return _loader

DatasetCatalog.register("pi_train_panoptic", _with_semseg("pi_train_coco", PI_TRAIN_MASKS))
DatasetCatalog.register("pi_val_panoptic",   _with_semseg("pi_val_coco",   PI_VAL_MASKS))

for name in ["pi_train_panoptic","pi_val_panoptic"]:
    MetadataCatalog.get(name).thing_classes = ["juniper"]
    MetadataCatalog.get(name).stuff_classes = ["background", "juniper"]

print("Datasets panoptic registrados: pi_train_panoptic, pi_val_panoptic")

In [ ]:
#@title 3. Configuración y entrenamiento (Train/Val PI; Test sólo inferencia)
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader, DatasetMapper
import detectron2.data.transforms as T

def build_panoptic_cfg(out_dir=OUT_PAN):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml"))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml")

    # Train/Val con datasets que incluyen semántica:
    cfg.DATASETS.TRAIN = ("pi_train_panoptic",)
    cfg.DATASETS.TEST  = ("pi_val_panoptic",)   # eval periódica en Val

    # Clases / cabezas
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1      # 1 clase de instancia: juniper
    cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 2   # stuff: background, juniper
    cfg.MODEL.SEM_SEG_HEAD.LOSS_WEIGHT = 0.5
    cfg.MODEL.ROI_MASK_HEAD.POOLER_RESOLUTION = 28
    cfg.MODEL.ROI_MASK_HEAD.CLS_AGNOSTIC_MASK = True

    # Anchors
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[12], [24], [48], [96], [192]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [0.5, 1.0, 2.0]

    # RPN/ROI
    cfg.MODEL.RPN.NMS_THRESH = 0.6
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST  = 400
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.30
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 384

    # Solver
    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = 0.0025
    cfg.SOLVER.MAX_ITER = 12000
    cfg.SOLVER.LR_SCHEDULER_NAME = "WarmupCosineLR"
    cfg.SOLVER.WARMUP_METHOD = "linear"
    cfg.SOLVER.WARMUP_ITERS = 1200
    cfg.SOLVER.STEPS = []

    cfg.SOLVER.AMP.ENABLED = True
    cfg.SOLVER.CLIP_GRADIENTS.ENABLED = True
    cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"
    cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0

    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.CHECKPOINT_PERIOD = 1000
    cfg.TEST.EVAL_PERIOD = 1000

    # Opcionales de robustez para semántica
    cfg.MODEL.SEM_SEG_HEAD.IGNORE_VALUE = 255
    cfg.INPUT.MASK_FORMAT = "bitmask"

    cfg.OUTPUT_DIR = out_dir
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg

class JuniperTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        augs = [
            T.RandomFlip(prob=0.5, horizontal=True,  vertical=False),
            T.RandomFlip(prob=0.5, horizontal=False, vertical=True),
            T.RandomRotation(angle=[0, 90, 180, 270], sample_style="choice"),
            T.ResizeShortestEdge(short_edge_length=[320,352,384,416,448,480,512], max_size=768, sample_style="choice"),
            T.RandomCrop("relative_range", (0.7, 0.7)),
            T.RandomBrightness(0.85, 1.15),
            T.RandomContrast(0.85, 1.15),
            T.RandomSaturation(0.85, 1.15),
        ]
        mapper = DatasetMapper(cfg, is_train=True, augmentations=augs, image_format="BGR")
        return build_detection_train_loader(cfg, mapper=mapper)

cfg = build_panoptic_cfg()
DO_TRAIN = False
if DO_TRAIN:
    trainer = JuniperTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

# Pesos finales; después se hará inferencia en PI/FW Test
final_ckpt = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
if os.path.exists(final_ckpt):
    cfg.MODEL.WEIGHTS = final_ckpt

In [ ]:
#@title 4. Predictors y fusión (cache)
from detectron2.engine import DefaultPredictor

def make_predictor(base_cfg, weights_path, nms=0.25, score_thr=0.0, name="pred"):
    c = base_cfg.clone()
    c.MODEL.WEIGHTS = weights_path
    c.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_thr
    c.MODEL.ROI_HEADS.NMS_THRESH_TEST   = nms
    pred = DefaultPredictor(c)
    pred._cache_id = f"{name}|{os.path.basename(weights_path)}|nms={nms}"
    return pred

WEIGHTS = cfg.MODEL.WEIGHTS
predictor_pi = make_predictor(cfg, WEIGHTS, nms=0.25, score_thr=0.0, name="pi")
predictor_fw = make_predictor(cfg, WEIGHTS, nms=0.25, score_thr=0.0, name="fw")

# Caché RAW
RAW_CACHE = {}

def _cache_key(predictor, img_path):
    return f"{getattr(predictor,'_cache_id','pred')}::{img_path}"

@torch.no_grad()
def _predict_raw_cached(img_bgr, predictor, img_path):
    """
    Ejecuta el predictor una única vez por imagen y guarda el resultado en caché.
    """
    key = _cache_key(predictor, img_path)
    if key in RAW_CACHE:
        return RAW_CACHE[key]
    out = predictor(img_bgr)
    inst = out["instances"].to("cpu")
    H, W = img_bgr.shape[:2]
    masks  = inst.pred_masks.numpy().astype(bool) if inst.has("pred_masks") else np.zeros((0, H, W), dtype=bool)
    scores = inst.scores.numpy() if inst.has("scores") else np.zeros((0,), dtype=float)
    RAW_CACHE[key] = (masks, scores)
    return masks, scores

def clear_raw_cache():
    global RAW_CACHE
    RAW_CACHE = {}
    print("RAW_CACHE limpiada")

def _bbox_from_mask(m):
    ys = np.any(m, axis=1); xs = np.any(m, axis=0)
    if not ys.any() or not xs.any(): return (0,0,0,0)
    y_idx = np.where(ys)[0]; x_idx = np.where(xs)[0]
    return (int(y_idx[0]), int(x_idx[0]), int(y_idx[-1]), int(x_idx[-1]))

def _box_iou(b1, b2):
    y1 = max(b1[0], b2[0]); x1 = max(b1[1], b2[1])
    y2 = min(b1[2], b2[2]); x2 = min(b1[3], b2[3])
    inter = max(0, y2-y1+1) * max(0, x2-x1+1)
    a1 = (b1[2]-b1[0]+1)*(b1[3]-b1[1]+1)
    a2 = (b2[2]-b2[0]+1)*(b2[3]-b2[1]+1)
    union = a1 + a2 - inter
    return inter/union if union>0 else 0.0

def _merge_by_iou_fast(masks_sorted, scores_sorted, iou_thr=0.85, downsample=2):
    N = masks_sorted.shape[0]
    if N <= 1: return masks_sorted, scores_sorted
    ms_small = masks_sorted[:, ::downsample, ::downsample] if (downsample and downsample > 1) else masks_sorted
    boxes = [ _bbox_from_mask(m) for m in masks_sorted ]
    kept_idx, kept_boxes, kept_small = [], [], []
    for i in range(N):
        b = boxes[i]; m_small = ms_small[i]; keep = True
        for j in range(len(kept_idx)):
            if _box_iou(b, kept_boxes[j]) < iou_thr: continue
            inter = np.logical_and(m_small, kept_small[j]).sum()
            uni   = np.logical_or(m_small, kept_small[j]).sum()
            if (inter/uni if uni>0 else 0.0) >= iou_thr:
                keep = False; break
        if keep:
            kept_idx.append(i); kept_boxes.append(b); kept_small.append(m_small)
    kept_idx = np.asarray(kept_idx, dtype=int)
    return masks_sorted[kept_idx], scores_sorted[kept_idx]


def predict_instances(img_bgr, predictor, img_path,
                      iou_merge=0.85,
                      merge_downsample=2,
                      topk=250):
    """
    Obtiene instancias a partir de una única pasada del predictor
    y aplica un merge rápido por IoU para eliminar duplicados solapados.
    Devuelve:
      - masks: np.ndarray (N, H, W) de bool
      - scores: np.ndarray (N,) de floats
    """
    H, W = img_bgr.shape[:2]

    # Predicción única
    masks0, scores0 = _predict_raw_cached(img_bgr, predictor, img_path)

    masks = masks0
    scores = scores0
    if masks.shape[0] == 0:
        return masks, scores

    # Ordenar por score y merge rápido por IoU
    order = np.argsort(-scores)
    ms, ss = masks[order], scores[order]
    ms = ms[:topk]
    ss = ss[:topk]
    ms_merged, ss_merged = _merge_by_iou_fast(ms, ss, iou_thr=iou_merge, downsample=merge_downsample)
    return ms_merged, ss_merged

In [ ]:
#@title 5. Utilidades de evaluación (IoU, S-IoU, pixelares, tablas/curvas)
def iou_matrix(pred_masks, gt_masks):
    if len(pred_masks)==0 or len(gt_masks)==0:
        return np.zeros((len(pred_masks), len(gt_masks)), dtype=float)
    P, G = len(pred_masks), len(gt_masks)
    M = np.zeros((P, G), dtype=float)
    for i, p in enumerate(pred_masks):
        ip = p.astype(bool); p_area = ip.sum()
        for j, g in enumerate(gt_masks):
            ig = g.astype(bool)
            inter = np.logical_and(ip, ig).sum()
            union = p_area + ig.sum() - inter
            M[i, j] = float(inter / max(1, union))
    return M

def eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=0.5):
    M = iou_matrix(pred_masks, gt_masks)
    if M.size == 0:
        return dict(tp=0, fp=len(pred_masks), fn=len(gt_masks))
    cost = 1.0 - M
    ri, cj = linear_sum_assignment(cost)
    matched_pred = set(); matched_gt = set()
    for i, j in zip(ri, cj):
        if M[i, j] >= iou_thr:
            matched_pred.add(i); matched_gt.add(j)
    tp = len(matched_pred); fp = len(pred_masks)-tp; fn = len(gt_masks)-tp
    return dict(tp=tp, fp=fp, fn=fn)

def siou_pred(p_mask, gt_masks):
    matches = [g for g in gt_masks if np.any(p_mask & g)]
    if not matches: return 0.0
    union_gt = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(p_mask, union_gt).sum()
    den = union_gt.sum()
    return float(inter/den) if den>0 else 0.0

def siou_label(g_mask, pred_masks):
    matches = [p for p in pred_masks if np.any(g_mask & p)]
    if not matches: return 0.0
    union_p = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(g_mask, union_p).sum()
    den = g_mask.sum()
    return float(inter/den) if den>0 else 0.0

def eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=0.5):
    order = np.argsort(-np.asarray(pred_scores)) if len(pred_scores)>0 else np.arange(len(pred_masks))
    tp = fp = 0
    for i in order:
        s = siou_pred(pred_masks[i], gt_masks)
        if s >= siou_thr: tp += 1
        else:             fp += 1
    fn = 0
    for g in gt_masks:
        s = siou_label(g, pred_masks)
        if s < siou_thr: fn += 1
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

def pixel_metrics(pred_binary: np.ndarray, gt_binary: np.ndarray):
    y_pred = pred_binary.astype(np.uint8).ravel()
    y_true = gt_binary.astype(np.uint8).ravel()
    tp = int(np.sum((y_true==1) & (y_pred==1)))
    tn = int(np.sum((y_true==0) & (y_pred==0)))
    fp = int(np.sum((y_true==0) & (y_pred==1)))
    fn = int(np.sum((y_true==1) & (y_pred==0)))
    total = tp + tn + fp + fn
    acc = (tp + tn) / total if total > 0 else 0.0
    iou_fg = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    iou_bg = tn / (tn + fp + fn) if (tn + fp + fn) > 0 else 0.0
    miou  = 0.5 * (iou_fg + iou_bg)
    w0 = (tn + fp) / total if total > 0 else 0.0
    w1 = (tp + fn) / total if total > 0 else 0.0
    fwiou = w0 * iou_bg + w1 * iou_fg
    return dict(pACC=acc, mIoU=miou, fwIoU=fwiou)

def counts_to_metrics(d):
    tp,fp,fn = d["tp"], d["fp"], d["fn"]
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

def save_curves_4(grid, f1_iou05, f1_iou075, f1_siou05, f1_siou075, tag, theta_mark=None):
    df = pd.DataFrame({
        "theta": grid,
        "F1_IoU_0.5":  (np.array(f1_iou05)*100.0 if f1_iou05 is not None else np.nan),
        "F1_IoU_0.75": (np.array(f1_iou075)*100.0 if f1_iou075 is not None else np.nan),
        "F1_SIoU_0.5": (np.array(f1_siou05)*100.0 if f1_siou05 is not None else np.nan),
        "F1_SIoU_0.75":(np.array(f1_siou075)*100.0 if f1_siou075 is not None else np.nan),
    })
    os.makedirs(os.path.join(OUT_PAN, "curves"), exist_ok=True)
    os.makedirs(os.path.join(OUT_PAN, "plots"), exist_ok=True)

    csv_path = os.path.join(OUT_PAN, "curves", f"curves_F1_vs_theta_{tag}_4curves.csv")
    df.to_csv(csv_path, index=False)

    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "legend.fontsize": 10, "figure.dpi": 150
    })
    fig, ax = plt.subplots(figsize=(6.7, 4.2))
    ax.plot(df["theta"], df["F1_IoU_0.5"],   marker="o",  label="IoU @ 0.5")
    ax.plot(df["theta"], df["F1_IoU_0.75"],  marker="s",  label="IoU @ 0.75")
    ax.plot(df["theta"], df["F1_SIoU_0.5"],  marker="^",  label="S-IoU @ 0.5")
    ax.plot(df["theta"], df["F1_SIoU_0.75"], marker="D",  label="S-IoU @ 0.75")
    if theta_mark is not None:
        ax.axvline(float(theta_mark), linestyle="--", linewidth=1)
    ax.set_xlabel("θ_score"); ax.set_ylabel("F1-score (%)")
    ax.set_title(f"{tag} — F1 vs θ_score (IoU y S-IoU) — Panoptic FPN")
    ax.grid(True, alpha=0.3); ax.legend(loc="best", frameon=False)

    png_path = os.path.join(OUT_PAN, "plots", f"F1_vs_theta_{tag}_4curves.png")
    plt.savefig(png_path, bbox_inches="tight"); plt.close()
    return csv_path, png_path

In [ ]:
#@title 6. Evaluación principal: curvas θ, θ*, tablas global/tamaño, pixelares (y guardado de máscaras)
def evaluate_panoptic_vs_coco(coco_json, img_dir, predictor, theta_grid, tag):
    coco = COCO(coco_json); images = coco.loadImgs(coco.getImgIds())
    curves_counts = {
        ("IoU",0.5):  [dict(tp=0,fp=0,fn=0) for _ in theta_grid],
        ("IoU",0.75): [dict(tp=0,fp=0,fn=0) for _ in theta_grid],
        ("S-IoU",0.5):[dict(tp=0,fp=0,fn=0) for _ in theta_grid],
        ("S-IoU",0.75):[dict(tp=0,fp=0,fn=0) for _ in theta_grid],
    }
    for im in tqdm(images, desc=f"Curvas {tag}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            continue
        H,W = img.shape[:2]
        gt_masks = ann_masks_for_img(coco, im["id"], H, W)

        pm_all, ps_all = predict_instances(img, predictor, fpath)

        for tidx, th in enumerate(theta_grid):
            pm = [m for m,s in zip(pm_all, ps_all) if s >= float(th)]
            ps = [s for s in ps_all if s >= float(th)]

            # IoU
            for t in (0.5, 0.75):
                m = eval_iou_at_threshold_hungarian(pm, gt_masks, iou_thr=t)
                for k in ("tp","fp","fn"): curves_counts[("IoU",t)][tidx][k] += m[k]

            # S-IoU
            for t in (0.5, 0.75):
                m = eval_siou_at_threshold(pm, ps, gt_masks, siou_thr=t)
                for k in ("tp","fp","fn"): curves_counts[("S-IoU",t)][tidx][k] += m[k]

    def _f1_list(key):
        return [counts_to_metrics(d)["f1"] for d in curves_counts[key]]

    return dict(
        thr_grid=list(map(float, theta_grid)),
        f1_iou05=_f1_list(("IoU",0.5)),
        f1_iou075=_f1_list(("IoU",0.75)),
        f1_siou05=_f1_list(("S-IoU",0.5)),
        f1_siou075=_f1_list(("S-IoU",0.75)),
    )

def pick_best_theta(curves_res, key="S-IoU"):
    grid = np.array(curves_res["thr_grid"])
    f1 = np.array(curves_res["f1_siou05" if key=="S-IoU" else "f1_iou05"])
    i = int(np.argmax(f1))
    return float(grid[i]), float(f1[i])

def evaluate_at_theta_and_save(coco_json, img_dir, predictor, theta_star, write_pred_masks_dir, tag):
    coco = COCO(coco_json); images = coco.loadImgs(coco.getImgIds())
    res_counts = {("IoU",0.5): dict(tp=0,fp=0,fn=0),
                  ("IoU",0.75): dict(tp=0,fp=0,fn=0),
                  ("S-IoU",0.5): dict(tp=0,fp=0,fn=0),
                  ("S-IoU",0.75): dict(tp=0,fp=0,fn=0)}
    sizewise = {name: {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)} for name,_,_ in SIZE_BINS}
    sizewise["All"] = {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)}
    pix_agg = dict(pACC=[], mIoU=[], fwIoU=[])
    os.makedirs(write_pred_masks_dir, exist_ok=True)

    for im in tqdm(images, desc=f"Eval@θ={theta_star:.2f} {tag}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath);
        if img is None: continue
        H,W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=GSD_FALLBACK)
        gt_masks = ann_masks_for_img(coco, im["id"], H, W)

        pm_all, ps_all = predict_instances(img, predictor, fpath)
        pm = [m for m,s in zip(pm_all, ps_all) if s >= float(theta_star)]
        ps = [s for s in ps_all if s >= float(theta_star)]

        # Guardar unión binaria
        pred_bin = np.any(np.stack(pm, axis=0), axis=0) if len(pm)>0 else np.zeros((H,W), bool)
        out_name = os.path.basename(fpath).replace("Img_","Mask_")
        with rasterio.open(fpath) as src:
            meta = src.meta.copy(); meta.update(count=1, dtype=rasterio.uint8, nodata=0)
        with rasterio.open(os.path.join(write_pred_masks_dir, out_name), "w", **meta) as dst:
            dst.write(pred_bin.astype(np.uint8), 1)

        # Pixelares
        gt_bin = np.any(np.stack(gt_masks,0),0) if len(gt_masks)>0 else np.zeros((H,W), bool)
        pm_pix = pixel_metrics(pred_bin, gt_bin)
        for k,v in pm_pix.items(): pix_agg[k].append(v)

        # Globales
        for t in (0.5, 0.75):
            m = eval_iou_at_threshold_hungarian(pm, gt_masks, iou_thr=t)
            for k in ("tp","fp","fn"): res_counts[("IoU",t)][k] += m[k]
            m = eval_siou_at_threshold(pm, ps, gt_masks, siou_thr=t)
            for k in ("tp","fp","fn"): res_counts[("S-IoU",t)][k] += m[k]

        # Tamaños (IoU@0.5)
        iou_mat = iou_matrix(pm, gt_masks) if (len(pm)>0 and len(gt_masks)>0) else np.zeros((len(pm), len(gt_masks)))
        assigned=set(); gt_areas = [float(g.sum())*(gsd**2) for g in gt_masks] if len(gt_masks)>0 else []
        if iou_mat.size>0:
            order = np.argsort(-iou_mat.max(axis=1))
            for i in order:
                if iou_mat.shape[1]==0: break
                j = int(np.argmax(iou_mat[i]))
                if iou_mat[i,j] >= 0.5 and j not in assigned:
                    assigned.add(j)
                    bname = size_label(gt_areas[j])
                    sizewise[bname][("IoU",0.5)]["tp"] += 1; sizewise["All"][("IoU",0.5)]["tp"] += 1
            for j in range(len(gt_masks)):
                if j not in assigned:
                    bname = size_label(gt_areas[j]); sizewise[bname][("IoU",0.5)]["fn"] += 1; sizewise["All"][("IoU",0.5)]["fn"] += 1
        for i in range(len(pm)):
            if iou_mat.shape[1]==0 or iou_mat[i].max() < 0.5:
                bname = size_label(float(pm[i].sum())*(gsd**2))
                sizewise[bname][("IoU",0.5)]["fp"] += 1; sizewise["All"][("IoU",0.5)]["fp"] += 1

        # Tamaños (S-IoU@0.5)
        for p in pm:
            overlaps = [g for g in gt_masks if np.any(p & g)]
            if not overlaps: continue
            union_gt = np.any(np.stack(overlaps, axis=0), axis=0)
            bname = size_label(float(union_gt.sum())*(gsd**2))
            s_pred = siou_pred(p, gt_masks)
            if s_pred >= 0.5:
                sizewise[bname][("S-IoU",0.5)]["tp"] += 1
                sizewise["All"][("S-IoU",0.5)]["tp"] += 1
            else:
                sizewise[bname][("S-IoU",0.5)]["fp"] += 1
                sizewise["All"][("S-IoU",0.5)]["fp"] += 1
        for g in gt_masks:
            s_lab = siou_label(g, pm)
            if s_lab < 0.5:
                bname = size_label(float(g.sum())*(gsd**2))
                sizewise[bname][("S-IoU",0.5)]["fn"] += 1
                sizewise["All"][("S-IoU",0.5)]["fn"] += 1

    metrics = {k:counts_to_metrics(v) for k,v in res_counts.items()}
    pix_summary = {k: float(np.mean(v)) if len(v)>0 else 0.0 for k,v in pix_agg.items()}
    size_metrics = {sname:{key:counts_to_metrics(cnt) for key,cnt in sub.items()} for sname,sub in sizewise.items()}
    return dict(metrics=metrics, pixel_metrics=pix_summary, size_metrics=size_metrics)

def to_df_global(res, data_tag, theta):
    rows=[]
    for (metric,thr),d in res["metrics"].items():
        rows.append(dict(Data=f"{data_tag} (θ={theta:.2f})", Metric=metric, Thr=thr,
                         TP=d["tp"], FP=d["fp"], FN=d["fn"],
                         Precision=100*d["precision"], Recall=100*d["recall"], F1_score=100*d["f1"]))
    return pd.DataFrame(rows)

def to_df_sizes(res, data_tag):
    def f1(p,r): return 100*(2*p*r/max(1e-9,(p+r)) if (p+r)>0 else 0.0)
    rows=[]
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = res["size_metrics"][s][("IoU",0.5)]; rs = res["size_metrics"][s][("S-IoU",0.5)]
        rows.append({"Data":data_tag,"Size":s,
                     "IoU_P":100*ri["precision"], "IoU_R":100*ri["recall"], "IoU_F1":f1(ri["precision"],ri["recall"]),
                     "S-IoU_P":100*rs["precision"], "S-IoU_R":100*rs["recall"], "S-IoU_F1":f1(rs["precision"],rs["recall"])})
    return pd.DataFrame(rows)

In [ ]:
#@title 7. Cobertura y densidad + Calibrado WS en FW
import os, math, numpy as np, rasterio
from scipy import ndimage
from sklearn.metrics import mean_squared_error, r2_score

# Utilidad mínima si no la tienes ya definida arriba
def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

# WS opcional (skimage). Fallback a CC si no disponible.
try:
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    from skimage.morphology import h_minima
    _SKIMAGE_OK = True
except Exception:
    _SKIMAGE_OK = False
    peak_local_max = watershed = h_minima = None
    print("scikit-image no disponible: WS usa fallback por CC.")

def split_ws_pred(mask_bin, min_dist_px=3, h_rel=0.10):
    if not _SKIMAGE_OK:
        labels, _ = ndimage.label(mask_bin.astype(np.uint8))
        return labels
    from scipy import ndimage as ndi
    mask_bin = mask_bin.astype(np.uint8)
    dist = ndi.distance_transform_edt(mask_bin)
    dist_supp = dist
    if dist.max() > 0 and h_rel > 0:
        try:
            dist_supp = dist - h_minima(dist, h=float(h_rel * dist.max()))
        except Exception:
            pass
    coords = peak_local_max(dist_supp, min_distance=int(max(1, min_dist_px)), labels=mask_bin)
    markers = np.zeros_like(mask_bin, dtype=np.int32)
    for i, (r, c) in enumerate(coords, start=1):
        markers[r, c] = i
    labels = watershed(-dist_supp, markers, mask=mask_bin)
    return labels

def density_cc(mask_bin):
    return int(ndimage.label(mask_bin.astype(np.uint8))[1])

def density_ws(mask_bin, min_dist_px=3, h_rel=0.10):
    labels = split_ws_pred(mask_bin.astype(np.uint8), min_dist_px=min_dist_px, h_rel=h_rel)
    return int(labels.max())

def coverage_density_from_folders(gt_dir, pr_dir, ws_params=None):
    """
    Calcula cobertura (fracción 0–1) y densidad (conteo por imagen) para cada par GT/Pred.
    - La cobertura se computa SOLO sobre píxeles válidos (NoData enmascarado por GT).
    - Devuelve métricas de cobertura en **%**: RMSE/MAE/MBE en puntos porcentuales + R².
    - Devuelve métricas de densidad (conteo por imagen) en unidades absolutas + R².
    - 'series' mantiene cov_T/cov_P en fracción (0–1) para compatibilidad con figuras.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cov_T, cov_P, den_T, den_P = [], [], [], []

    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)

        # Enmascarado de NoData desde el GT
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, dtype=bool)
        if valid.sum() == 0:
            continue

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        # Cobertura (fracción 0–1) solo en píxeles válidos
        cov_T.append(float(gtb.sum() / valid.sum()))
        cov_P.append(float(prb.sum() / valid.sum()))

        # Densidad (conteo por imagen)
        den_T.append(density_cc(gtb))
        if ws_params is None:
            den_P.append(density_cc(prb))
        else:
            den_P.append(density_ws(prb,
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10))))

    # Métricas de cobertura en % (puntos porcentuales)
    if cov_T:
        rmse_cover = float(np.sqrt(mean_squared_error(cov_T, cov_P))) * 100.0
        mae_cover  = float(np.mean(np.abs(np.array(cov_T) - np.array(cov_P)))) * 100.0
        mbe_cover  = float(np.mean(np.array(cov_P) - np.array(cov_T))) * 100.0
        r2_cover   = float(r2_score(cov_T, cov_P))
    else:
        rmse_cover = mae_cover = mbe_cover = r2_cover = float("nan")

    # Métricas de densidad (conteo por imagen)
    if den_T:
        rmse_density = float(np.sqrt(mean_squared_error(den_T, den_P)))
        mae_density  = float(np.mean(np.abs(np.array(den_T) - np.array(den_P))))
        mbe_density  = float(np.mean(np.array(den_P) - np.array(den_T)))
        r2_density   = float(r2_score(den_T, den_P))
    else:
        rmse_density = mae_density = mbe_density = r2_density = float("nan")

    return (
        dict(
            N=len(cov_T),
            RMSE_cover=rmse_cover, MAE_cover=mae_cover, MBE_cover=mbe_cover, R2_cover=r2_cover,
            RMSE_density=rmse_density, MAE_density=mae_density, MBE_density=mbe_density, R2_density=r2_density
        ),
        dict(cov_T=cov_T, cov_P=cov_P, den_T=den_T, den_P=den_P)
    )

# Área válida (ha) y densidad por hectárea
try:
    from pyproj import Geod
    _HAS_PYPROJ = True
    _GEOD = Geod(ellps="WGS84")
except Exception:
    _HAS_PYPROJ = False
    _GEOD = None

def _poly_area_m2_from_bounds(l, b, r, t):
    if not _HAS_PYPROJ:
        return None
    lons = [l, l, r, r, l]; lats = [b, t, t, b, b]
    area, _ = _GEOD.polygon_area_perimeter(lons, lats)
    return abs(area)

def _m_per_deg_lat(lat_rad):
    return (111132.92 - 559.82*math.cos(2*lat_rad) + 1.175*math.cos(4*lat_rad) - 0.0023*math.cos(6*lat_rad))

def _m_per_deg_lon(lat_rad):
    return (111412.84*math.cos(lat_rad) - 93.5*math.cos(3*lat_rad) + 0.118*math.cos(5*lat_rad))

def raster_valid_area_ha(img_path, valid_mask=None):
    with rasterio.open(img_path) as src:
        H, W = src.height, src.width
        tr, crs, bounds = src.transform, src.crs, src.bounds
        nodata = src.nodata
        if valid_mask is None:
            try:
                arr = src.read(1)
                valid_mask = (arr != nodata) if nodata is not None else np.ones((H, W), dtype=bool)
            except Exception:
                valid_mask = np.ones((H, W), dtype=bool)
        valid_px = int(np.sum(valid_mask))
        if valid_px == 0:
            return 0.0

        # proyectado → área por determinante de la transformada afín
        det = abs(tr.a * tr.e - tr.b * tr.d)
        if crs is not None and getattr(crs, "is_projected", False):
            return (det * valid_px) / 10000.0

        # geodésico si hay pyproj
        area_geo = _poly_area_m2_from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top)
        if area_geo is not None:
            return (area_geo * (valid_px / (H * W))) / 10000.0

        # aproximación métrica por grado
        lat_c = 0.5 * (bounds.bottom + bounds.top); lat_rad = math.radians(lat_c)
        mx = _m_per_deg_lon(lat_rad); my = _m_per_deg_lat(lat_rad)
        px_m2 = (mx * tr.a) * (my * abs(tr.e))
        return (valid_px * px_m2) / 10000.0

def eval_density_per_ha(gt_dir, pr_dir, ws_params=None):
    """
    Densidad (ind/ha), usando área válida (NoData enmascarado) y WS opcional.
    Devuelve y_true/y_pred + RMSE y R² en ind/ha.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    yT, yP = [], []

    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, bool)
        area_ha = max(raster_valid_area_ha(gt_path, valid_mask=valid), 1e-9)

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        labs_gt, _ = ndimage.label(gtb.astype(np.uint8))
        dens_gt = int(labs_gt.max()) / area_ha

        if ws_params is None:
            labs_pr, _ = ndimage.label(prb.astype(np.uint8))
            dens_pr = int(labs_pr.max()) / area_ha
        else:
            labs_pr = split_ws_pred(prb.astype(np.uint8),
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10)))
            dens_pr = int(labs_pr.max()) / area_ha

        yT.append(dens_gt); yP.append(dens_pr)

    rmse = float(np.sqrt(mean_squared_error(yT, yP))) if yT else np.nan
    r2   = float(r2_score(yT, yP)) if yT else np.nan
    return dict(y_true=yT, y_pred=yP, rmse=rmse, r2=r2)

def calibrate_ws_density(gt_dir, pr_dir,
                         grid_min_dist=(2,3,4,5,6,7),
                         grid_h=(0.05,0.08,0.10,0.12,0.15,0.20)):
    """
    Búsqueda de (min_dist_px, h_rel) que minimiza RMSE del conteo por imagen (no por ha).
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    best = {"rmse": 1e9, "min_dist_px": None, "h_rel": None}
    for dmin in grid_min_dist:
        for h in grid_h:
            yT, yP = [], []
            for f in files:
                gt_path = os.path.join(gt_dir, f)
                pr_path = os.path.join(pr_dir, f)
                if not os.path.exists(pr_path):
                    alt = f.replace("Img_", "Mask_"); pr_path = os.path.join(pr_dir, alt)
                    if not os.path.exists(pr_path):
                        continue
                gt = read_mask(gt_path); pr = read_mask(pr_path)
                gtb = (gt == 1); prb = (pr == 1)
                labs_gt, _ = ndimage.label(gtb.astype(np.uint8)); dens_gt = int(labs_gt.max())
                labs_pr = split_ws_pred(prb.astype(np.uint8), min_dist_px=int(dmin), h_rel=float(h))
                dens_pr = int(labs_pr.max())
                yT.append(dens_gt); yP.append(dens_pr)
            if yT:
                rmse = float(np.sqrt(mean_squared_error(yT, yP)))
                if rmse < best["rmse"]:
                    best = {"rmse": rmse, "min_dist_px": int(dmin), "h_rel": float(h)}
    return best

In [ ]:
#@title 8. Pipeline completo: curvas θ, θ*, evaluación, pixelares, cobertura/densidad, WS, guardados

theta_grid = np.linspace(0.05, 0.95, 19)

# Curvas y θ* (PI)
clear_raw_cache()
print("PI (curvas vs θ_score)…")
curves_pi = evaluate_panoptic_vs_coco(PI_COCO_JSON, PI_IMG_DIR, predictor_pi, theta_grid, tag="Panoptic-PI")
theta_pi, f1_pi = pick_best_theta(curves_pi, key="S-IoU")
csv_pi, png_pi = save_curves_4(
    curves_pi["thr_grid"],
    f1_iou05=curves_pi["f1_iou05"],
    f1_iou075=curves_pi["f1_iou075"],
    f1_siou05=curves_pi["f1_siou05"],
    f1_siou075=curves_pi["f1_siou075"],
    tag="PI",
    theta_mark=theta_pi
)
print(f"PI: θ* (S-IoU@0.5) = {theta_pi:.2f} | F1 = {100*f1_pi:.2f}%")

# Curvas y θ* (FW)
clear_raw_cache()
print("FW (curvas vs θ_score)…")
curves_fw = evaluate_panoptic_vs_coco(FW_COCO_JSON, FW_IMG_DIR, predictor_fw, theta_grid, tag="Panoptic-FW")
theta_fw, f1_fw = pick_best_theta(curves_fw, key="S-IoU")
csv_fw, png_fw = save_curves_4(
    curves_fw["thr_grid"],
    f1_iou05=curves_fw["f1_iou05"],
    f1_iou075=curves_fw["f1_iou075"],
    f1_siou05=curves_fw["f1_siou05"],
    f1_siou075=curves_fw["f1_siou075"],
    tag="FW",
    theta_mark=theta_fw
)
print(f"FW: θ* (S-IoU@0.5) = {theta_fw:.2f} | F1 = {100*f1_fw:.2f}%")

# Evaluación a θ* y guardado de máscaras
print("\nEvaluando a θ* y guardando máscaras…")
res_pi = evaluate_at_theta_and_save(PI_COCO_JSON, PI_IMG_DIR, predictor_pi, theta_pi, PRED_PI, tag="PI")
res_fw = evaluate_at_theta_and_save(FW_COCO_JSON, FW_IMG_DIR, predictor_fw, theta_fw, PRED_FW, tag="FW")

df_pi = to_df_global(res_pi, "PI-test", theta_pi)
df_fw = to_df_global(res_fw, "FW-test", theta_fw)
df_all = pd.concat([df_pi, df_fw], ignore_index=True)
from IPython.display import display
display(df_all)
df_all.to_csv(f"{OUT_PAN}/csv/global_metrics_theta_star.csv", index=False)
with open(f"{OUT_PAN}/tables/table_global_theta_star.tex","w") as f:
    f.write(df_all.to_latex(index=False, float_format='%.6f'))

df_pi_sz = to_df_sizes(res_pi, "PI-test"); df_fw_sz = to_df_sizes(res_fw, "FW-test")
display(df_pi_sz); display(df_fw_sz)
df_pi_sz.to_csv(f"{OUT_PAN}/csv/sizewise_PI_theta_star.csv", index=False)
df_fw_sz.to_csv(f"{OUT_PAN}/csv/sizewise_FW_theta_star.csv", index=False)
with open(f"{OUT_PAN}/tables/table_sizewise_PI_theta_star.tex","w") as f:
    f.write(df_pi_sz.to_latex(index=False, float_format='%.2f'))
with open(f"{OUT_PAN}/tables/table_sizewise_FW_theta_star.tex","w") as f:
    f.write(df_fw_sz.to_latex(index=False, float_format='%.2f'))

# Pixelares (mIoU, pixAcc, fwIoU) a θ*
def miou_pixacc_fwIoU_folder(gt_dir, pred_dir, num_classes=2):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for f in files:
        gt_path = os.path.join(gt_dir, f); pr_path = os.path.join(pred_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_","Mask_"); pr_path = os.path.join(pred_dir, alt)
            if not os.path.exists(pr_path): continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        gt = (gt==1).astype(np.int64); pr = (pr==1).astype(np.int64)
        for i in range(num_classes):
            for j in range(num_classes):
                cm[i,j] += np.sum((gt==i)&(pr==j))
    tp = np.diag(cm); total = cm.sum()
    pixacc = float(tp.sum()/total) if total>0 else 0.0
    ious=[]
    for c in range(num_classes):
        denom = tp[c] + (cm[c,:].sum()-tp[c]) + (cm[:,c].sum()-tp[c])
        ious.append(float(tp[c]/denom) if denom>0 else 0.0)
    miou = float(np.mean(ious)) if ious else 0.0
    freq = cm.sum(axis=1)/total if total>0 else np.zeros(num_classes)
    fwiou = float((freq*np.array(ious)).sum())
    return dict(mIoU=miou, pixAcc=pixacc, fwIoU=fwiou)

pix_PI = miou_pixacc_fwIoU_folder(GT_PI, PRED_PI, num_classes=2)
pix_FW = miou_pixacc_fwIoU_folder(GT_FW, PRED_FW, num_classes=2)
df_pix = pd.DataFrame([
    dict(Split="PI-test", mIoU=pix_PI["mIoU"], pixAcc=pix_PI["pixAcc"], fwIoU=pix_PI["fwIoU"]),
    dict(Split="FW-test", mIoU=pix_FW["mIoU"], pixAcc=pix_FW["pixAcc"], fwIoU=pix_FW["fwIoU"]),
])
display(df_pix)
df_pix.to_csv(f"{OUT_PAN}/csv/pixel_metrics_theta_star.csv", index=False)
with open(f"{OUT_PAN}/tables/table_pixel_metrics_theta_star.tex","w") as f:
    f.write(df_pix.to_latex(index=False, float_format="%.6f"))

# Cobertura/Densidad por imagen (WS baseline = CC)
print("\nCobertura/Densidad por imagen (θ* | WS baseline=CC):")
WS_BASE = None
pi_cd, _ = coverage_density_from_folders(GT_PI, PRED_PI, ws_params=WS_BASE)
fw_cd, _ = coverage_density_from_folders(GT_FW, PRED_FW, ws_params=WS_BASE)
print("PI: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    pi_cd["RMSE_cover"], pi_cd["MAE_cover"], pi_cd["MBE_cover"], pi_cd["R2_cover"],
    pi_cd["RMSE_density"], pi_cd["MAE_density"], pi_cd["MBE_density"], pi_cd["R2_density"]))
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd["RMSE_cover"], fw_cd["MAE_cover"], fw_cd["MBE_cover"], fw_cd["R2_cover"],
    fw_cd["RMSE_density"], fw_cd["MAE_density"], fw_cd["MBE_density"], fw_cd["R2_density"]))

# Densidad por hectárea (WS baseline = CC)
print("\nDensidad por hectárea (θ* | WS baseline=CC)")
pi_ha = eval_density_per_ha(GT_PI, PRED_PI, ws_params=WS_BASE)
fw_ha = eval_density_per_ha(GT_FW, PRED_FW, ws_params=WS_BASE)
print("PI: RMSE={:.2f} ind/ha | R²={:.3f}".format(pi_ha["rmse"], pi_ha["r2"]))
print("FW: RMSE={:.2f} ind/ha | R²={:.3f}".format(fw_ha["rmse"], fw_ha["r2"]))

# Calibración WS en FW
print("\nCalibrando WS en FW…")
best_ws_fw = calibrate_ws_density(GT_FW, PRED_FW)
print("WS FW óptimo:", best_ws_fw)
WS_PARAMS_FW = dict(min_dist_px=int(best_ws_fw["min_dist_px"]), h_rel=float(best_ws_fw["h_rel"]))
fw_cd_cal, _ = coverage_density_from_folders(GT_FW, PRED_FW, ws_params=WS_PARAMS_FW)
fw_ha_cal = eval_density_per_ha(GT_FW, PRED_FW, ws_params=WS_PARAMS_FW)
print("\nCobertura/Densidad FW (θ* + WS calibrado):")
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd_cal["RMSE_cover"], fw_cd_cal["MAE_cover"], fw_cd_cal["MBE_cover"], fw_cd_cal["R2_cover"],
    fw_cd_cal["RMSE_density"], fw_cd_cal["MAE_density"], fw_cd_cal["MBE_density"], fw_cd_cal["R2_density"]))

print("\nDensidad por hectárea FW (θ* + WS calibrado):")
print("FW: RMSE={:.2f} ind/ha | R²={:.3f}".format(fw_ha_cal["rmse"], fw_ha_cal["r2"]))

# Guardados cobertura/densidad
df_covdens = pd.DataFrame([
    dict(Split="PI-test", **pi_cd, RMSE_dens_ha=pi_ha["rmse"], R2_dens_ha=pi_ha["r2"]),
    dict(Split="FW-test", **fw_cd, RMSE_dens_ha=fw_ha["rmse"], R2_dens_ha=fw_ha["r2"]),
])
df_covdens.to_csv(f"{OUT_PAN}/csv/coverage_density_theta_star.csv", index=False)
with open(f"{OUT_PAN}/tables/table_coverage_density_theta_star.tex","w") as f:
    f.write(df_covdens.to_latex(index=False, float_format="%.6f"))

df_fw_ws = pd.DataFrame([dict(
    Split="FW-test",
    WS_min_dist_px=WS_PARAMS_FW["min_dist_px"],
    WS_h_rel=WS_PARAMS_FW["h_rel"],
    RMSE_cover=fw_cd_cal["RMSE_cover"],
    R2_cover=fw_cd_cal["R2_cover"],
    RMSE_density=fw_cd_cal["RMSE_density"],
    R2_density=fw_cd_cal["R2_density"],
    RMSE_dens_ha=fw_ha_cal["rmse"],
    R2_dens_ha=fw_ha_cal["r2"]
)])
df_fw_ws.to_csv(f"{OUT_PAN}/csv/fw_ws_calibrated_summary.csv", index=False)
with open(f"{OUT_PAN}/tables/table_fw_ws_calibrated_summary.tex","w") as f:
    f.write(df_fw_ws.to_latex(index=False, float_format="%.6f"))

print("\nListo")
print(f"Curvas guardadas en: {csv_pi}  y  {csv_fw}")
print(f"Máscaras a θ*: {PRED_PI}  |  {PRED_FW}")
print(f"CSV/LaTeX en: {OUT_PAN}/csv  y  {OUT_PAN}/tables")

In [ ]:
#@title 9. Figuras extra: Scatter FW y Barras por tamaño (PI/FW)
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr

def _rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))) if len(y_true) else float("nan")
def _mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred))) if len(y_true) else float("nan")
def _mbe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(y_pred - y_true)) if len(y_true) else float("nan")

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 10, "figure.dpi": 150
})

# Usa WS calibrado si existe; si no, baseline (CC)
try:
    _WS = WS_PARAMS_FW
except NameError:
    _WS = None

# Series cobertura/densidad FW
_, series_fw = coverage_density_from_folders(GT_FW, PRED_FW, ws_params=_WS)
_cov_T = np.array(series_fw["cov_T"]) * 100.0
_cov_P = np.array(series_fw["cov_P"]) * 100.0

den_fw = eval_density_per_ha(GT_FW, PRED_FW, ws_params=_WS)
_den_T = np.array(den_fw["y_true"])
_den_P = np.array(den_fw["y_pred"])

fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# (a) Cobertura
r_cov = pearsonr(_cov_T, _cov_P)[0] if len(_cov_T) > 1 else np.nan
axs[0].scatter(_cov_T, _cov_P, s=12)
lim = [0, max(1e-6, _cov_T.max(), _cov_P.max()) * 1.05]
axs[0].plot(lim, lim, linestyle="--")
axs[0].set_xlim(lim); axs[0].set_ylim(lim)
axs[0].set_xlabel("Observed canopy cover (%)")
axs[0].set_ylabel("Predicted canopy cover (%)")
axs[0].set_title(f"(a) FW — Cover @ θ={theta_fw:.2f}")
axs[0].text(0.02, 0.98,
            f"r={r_cov:.2f}\nRMSE={_rmse(_cov_T,_cov_P):.2f}%\nMAE={_mae(_cov_T,_cov_P):.2f}%\nMBE={_mbe(_cov_T,_cov_P):.2f}%",
            transform=axs[0].transAxes, va="top")

# (b) Densidad
r_den = pearsonr(_den_T, _den_P)[0] if len(_den_T) > 1 else np.nan
axs[1].scatter(_den_T, _den_P, s=12)
lim2 = [0, max(1e-6, _den_T.max(), _den_P.max()) * 1.05]
axs[1].plot(lim2, lim2, linestyle="--")
axs[1].set_xlim(lim2); axs[1].set_ylim(lim2)
axs[1].set_xlabel("Observed shrubs per ha")
axs[1].set_ylabel("Predicted shrubs per ha")
axs[1].set_title(f"(b) FW — Density @ θ={theta_fw:.2f}")
axs[1].text(0.02, 0.98,
            f"r={r_den:.2f}\nRMSE={_rmse(_den_T,_den_P):.2f}\nMAE={_mae(_den_T,_den_P):.2f}\nMBE={_mbe(_den_T,_den_P):.2f}",
            transform=axs[1].transAxes, va="top")

fig.suptitle("Observed vs Predicted — FW — Panoptic FPN")
fig.tight_layout(rect=[0, 0, 1, 0.93])
scatter_path = os.path.join(OUT_PAN, "plots", "FW_scatter_cover_density_Panoptic.png")
plt.savefig(scatter_path, bbox_inches="tight"); plt.close()
print("Scatter FW guardado en:", scatter_path)

# Barras por tamaño (PI/FW)
def _df_sizewise_from_res(res, data_tag):
    def f1(p,r): return 100*(2*p*r/max(1e-9,(p+r)) if (p+r)>0 else 0.0)
    rows=[]
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = res["size_metrics"][s][("IoU",0.5)]
        rs = res["size_metrics"][s][("S-IoU",0.5)]
        rows.append({"Data":data_tag, "Size":s,
                     "IoU_F1":f1(ri["precision"], ri["recall"]),
                     "S-IoU_F1":f1(rs["precision"], rs["recall"])})
    return pd.DataFrame(rows)

try:
    df_pi_sz_plot = df_pi_sz.copy()
    assert {"Size","IoU_F1","S-IoU_F1"}.issubset(df_pi_sz_plot.columns)
except Exception:
    df_pi_sz_plot = _df_sizewise_from_res(res_pi, "PI-test")

try:
    df_fw_sz_plot = df_fw_sz.copy()
    assert {"Size","IoU_F1","S-IoU_F1"}.issubset(df_fw_sz_plot.columns)
except Exception:
    df_fw_sz_plot = _df_sizewise_from_res(res_fw, "FW-test")

order_bins = [b[0] for b in SIZE_BINS] + ["All"]
df_pi_sz_plot = df_pi_sz_plot.set_index("Size").reindex(order_bins).reset_index()
df_fw_sz_plot = df_fw_sz_plot.set_index("Size").reindex(order_bins).reset_index()

def _plot_size_bars(df_sz, split_name, out_name, theta_txt):
    labels = df_sz["Size"].tolist()
    x = np.arange(len(labels)); width = 0.38
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.bar(x - width/2, df_sz["IoU_F1"].values,  width, label="IoU @ 0.5")
    ax.bar(x + width/2, df_sz["S-IoU_F1"].values, width, label="S-IoU @ 0.5")
    ax.set_xticks(x, labels)
    ax.set_ylabel("F1-score (%)"); ax.set_xlabel("Size bin")
    ax.set_title(f"{split_name} — Size-wise F1 (θ={theta_txt}) — Panoptic FPN")
    ax.grid(axis="y", alpha=0.3); ax.legend(loc="best", frameon=False)
    out_path = os.path.join(OUT_PAN, "plots", out_name)
    plt.savefig(out_path, bbox_inches="tight"); plt.close()
    return out_path

pi_bars_path = _plot_size_bars(df_pi_sz_plot, "PI", "PI_sizewise_bars_Panoptic.png", f"{theta_pi:.2f}")
fw_bars_path = _plot_size_bars(df_fw_sz_plot, "FW", "FW_sizewise_bars_Panoptic.png", f"{theta_fw:.2f}")
print("Barras por tamaño guardadas en:")
print(" -", pi_bars_path)
print(" -", fw_bars_path)